In [15]:
# Librerias
import pandas as pd
import requests
from dotenv import load_dotenv
import os
import boto3
import json
from botocore.exceptions import ClientError, BotoCoreError


In [ ]:
#Cargar variables de entorno
load_dotenv()
API_KEY = os.getenv("OpenWeatherMap_API_KEY")

S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME")

#Creacion de cliente
s3 = boto3.client('s3')

#Función
def lambda_handler(event, context):
    # Extracción de data de OpenWeatherMap API
    city = 'Lima'
    url =  f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}"
    
    response = requests.get(url)
    json_data = response.json()
    
    try:
    # Transformacion de data
        weather = {
            'city': json_data['name'],
            'country': json_data['sys']['country'],
            'temperature': json_data['main']['temp'],
            'humidity': json_data['main']['humidity'],
            'pressure': json_data['main']['pressure'],
            'wind_speed': json_data['wind']['speed'],
            }
        
        file_name = f"{weather['city']}_{weather['country']}.json"
        
        
        #Subida del JSON a S3
        s3.put_object(
            Bucket=S3_BUCKET_NAME,
            Key=file_name,
            Body=json.dumps(weather),
            ContentType='application/json'
        )
        

        return {
            'statusCode': 200,
            'body': f"Weather data for {weather['city']} has been uploaded to S3 bucket {S3_BUCKET_NAME}"
        }
        
    except requests.exceptions.HTTPError as e:
        return {
            'statusCode': response.status_code,
            'body':f"Weather API HTTP error: {str(e)}"
        }
    
    except requests.exceptions.RequestException as e:
        return {
            'statusCode': 500,
            'body': f"Weather API request failed: {str(e)}"
        }
    
    except ClientError as e:
        return {
            'statusCode': 500,
            'body':f"S3 ClientError: {e.response['Error']['Message']}"
        }
    
    except BotoCoreError as e:
        return {
            'statusCode': 500,
            'body':f"S3 BotoCoreError: {str(e)}"
        }
    
    except Exception as e:
        return {
            'statusCode': 500,
            'body':f"Unexpected error: {str(e)}"
        }

    

print(lambda_handler(None, None))
